In [14]:
import numpy as np
import pandas as pd
import os
from collections import defaultdict

In [15]:
def _parse_npz(path):
    loaded_data_file = np.load(path)

    # 2. Reconstruct the dictionary
    reconstructed_data = {}
    for full_key in loaded_data_file.files:
        # Split the key (e.g., 'A/X' -> ('A', 'X'))
        outer_key, inner_key = full_key.split('/')

        # Get the array
        array = loaded_data_file[full_key]

        # Initialize the inner dict if it doesn't exist
        if outer_key not in reconstructed_data:
            reconstructed_data[outer_key] = {}

        # Store the array
        reconstructed_data[outer_key][inner_key] = array

    print("Data successfully loaded and reconstructed.")
    return reconstructed_data

def parse_results(path):
    return _parse_npz(os.path.join(path, 'accuracy_results.npz')), _parse_npz(os.path.join(path, 'full_results.npz'))

In [16]:
def dict_to_pandas(data):
    records = []

    for outer_key, inner_dict in data.items():
        # Initialize a temporary dictionary for the current record (row)
        record = {'test_mode': outer_key} # Use the outer key as a unique identifier

        for inner_key, np_array in inner_dict.items():
            # Crucial step: Extract the scalar value from the one-element array.
            # np_array.item() is the most robust way to get the scalar.
            scalar_value = np_array.item()
            
            # Add the inner key/value pair to the record
            record[inner_key] = scalar_value
        
        # Add the completed record to the list
        records.append(record)
    
    df = pd.DataFrame(records)
    df = df.set_index('test_mode')
    return df

In [17]:
path = 'experiments/exp2'
accuracy_results, full_results = parse_results(path)

Data successfully loaded and reconstructed.
Data successfully loaded and reconstructed.


In [18]:
accuracy_results_pd = dict_to_pandas(accuracy_results)
display(accuracy_results_pd)

,train:random,train:int,train:upper,train:lower,train:ortho,train:random_eps
test_mode,,,,,,
test:random,0.7034,0.6974,0.3726,0.3932,0.4026,0.7000
test:int,0.7046,0.6920,0.3792,0.4040,0.4024,0.6990
test:upper,0.5570,0.5856,0.9720,0.8274,0.4376,0.5538
test:lower,0.6506,0.6244,0.8570,0.9852,0.4056,0.6492
test:ortho,0.5274,0.4806,0.3872,0.4062,1.0000,0.4004
test:random_eps,0.6662,0.6846,0.3648,0.3000,0.3462,0.6942


In [21]:
def _compute_accuracy_from_cm(matrix):
    return np.trace(matrix) / np.sum(matrix)

def _compute_accuracies(full_results):
    accuracies = defaultdict(dict)
    for outer_k, v in full_results.items():
        for inner_k, matrix in v.items():
            accuracies[outer_k][inner_k] = _compute_accuracy_from_cm(matrix[1:,1:])
    df = dict_to_pandas(accuracies)
    return df

In [23]:
accuracies = _compute_accuracies(full_results)

In [25]:
with pd.option_context('display.precision', 3):
    display(accuracies)

,train:random,train:int,train:upper,train:lower,train:ortho,train:random_eps
test_mode,,,,,,
test:random,0.629,0.622,0.248,0.242,0.253,0.626
test:int,0.631,0.615,0.258,0.255,0.253,0.624
test:upper,0.452,0.492,0.976,0.793,0.298,0.465
test:lower,0.563,0.530,0.822,0.982,0.257,0.563
test:ortho,0.417,0.383,0.260,0.258,1.000,0.353
test:random_eps,0.620,0.624,0.269,0.255,0.255,0.626
